# Análisis del entrenamiento del carrito DQN

Ejecuta primero `train.py`. Este notebook no entrena: analiza el archivo `results/training_log.csv` y obliga a separar evidencia de interpretación.

## Preguntas previas

1. ¿Qué esperarías que ocurriera con epsilon?
2. ¿Una recompensa creciente garantiza cero colisiones?
3. ¿Por qué conviene usar una media móvil?

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

LOG_PATH = Path('results/training_log.csv')
if not LOG_PATH.exists():
    raise FileNotFoundError('Ejecuta primero: python train.py --episodes 600')
logs = pd.read_csv(LOG_PATH)
display(logs.head())
print(logs.shape)


In [ ]:
logs[['episode', 'reward', 'steps', 'success', 'collision', 'epsilon', 'loss_mean']].describe()


## Curvas de aprendizaje

La línea tenue muestra cada episodio; la línea gruesa resume 30 episodios. El suavizado ayuda a ver la tendencia, pero no debe ocultar la variabilidad.

In [ ]:
window = 30
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(logs.episode, logs.reward, alpha=.25, label='episodio')
axes[0].plot(logs.episode, logs.reward.rolling(window, min_periods=1).mean(), lw=2, label='media móvil')
axes[0].set_ylabel('Retorno')
axes[0].legend(); axes[0].grid(alpha=.2)
axes[1].plot(logs.episode, logs.success.rolling(window, min_periods=1).mean(), lw=2, color='teal')
axes[1].axhline(.90, ls='--', color='coral')
axes[1].set(xlabel='Episodio', ylabel='Tasa de éxito móvil', ylim=(-.02, 1.02))
axes[1].grid(alpha=.2)
plt.show()


## Contraste entre el inicio y el final

In [ ]:
n = min(50, len(logs)//2)
comparison = pd.DataFrame({
    'inicio': logs.head(n)[['reward','steps','success','collision']].mean(),
    'final': logs.tail(n)[['reward','steps','success','collision']].mean(),
})
comparison['cambio'] = comparison['final'] - comparison['inicio']
comparison


## Uso de las acciones

In [ ]:
action_columns = [column for column in logs.columns if column.startswith('action_')]
usage = logs[action_columns].sum().sort_values(ascending=False)
usage.plot(kind='bar', figsize=(10,4), color='#2F80ED')
plt.ylabel('Número de decisiones')
plt.title('Distribución de acciones durante el entrenamiento')
plt.grid(axis='y', alpha=.2)
plt.show()


## Interpretación final

Responde con datos: ¿aumentó el éxito?, ¿disminuyeron las colisiones?, ¿la política depende demasiado de una acción?, ¿qué episodio marca un cambio sostenido?, ¿el modelo conserva su desempeño al evaluar sin exploración?